# Robustness of Accessibility
## Notebook 2/4 - create disrupted graph based on normal graph and hazard data

In [1]:
import geopandas as gpd
from shapely.ops import unary_union
import osmnx as ox

In [2]:
import os
from pathlib import Path

# Repo-Root finden & als Working Directory setzen
p = Path.cwd().resolve()
while not (p / "css_geodata_service").exists() and p.parent != p:
    p = p.parent
os.chdir(p)
print("CWD reset to repo root:", Path.cwd())

from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    RoaNotebookConfig,
    get_roa_cache_path,
    get_roa_inputs_path,
)

event = RoaNotebookConfig.event

inputs_dir: Path = get_roa_inputs_path()
cache_dir: Path = get_roa_cache_path()

print("inputs_dir =", inputs_dir)
print("cache_dir  =", cache_dir)

CWD reset to repo root: C:\Users\baiis\Desktop\roa_fopra
Working dir set to: C:\Users\baiis\Desktop\roa_fopra
Working dir set to: C:\Users\baiis\Desktop\roa_fopra
inputs_dir = C:\Users\baiis\Desktop\roa_fopra\css_geodata_service\robustness_of_accessibility\data\input
cache_dir  = C:\Users\baiis\Desktop\roa_fopra\css_geodata_service\robustness_of_accessibility\data\processed


In [3]:
undirected_graph = RoaNotebookConfig.undirected_graph
place_name = RoaNotebookConfig.place_name
show_all_plots = False

In [4]:
from shapely.geometry import Point

# Evacuation circle around bomb discovery point
bomb_lon = 6.660376
bomb_lat = 49.741604
radius_m = 1000

# Punkt in WGS84
pt = gpd.GeoSeries([Point(bomb_lon, bomb_lat)], crs="EPSG:4326")

# Projektion nach UTM (meter-basiert) und Buffer dort
pt_utm = pt.to_crs("EPSG:32632")  # Trier liegt in UTM Zone 32N
evacuation_area_utm = pt_utm.buffer(radius_m)

# Zurück nach WGS84 für Plot/Folium
evacuation_area_gdf = evacuation_area_utm.to_crs("EPSG:4326")
evacuation_area = evacuation_area_gdf.iloc[0]

### Load default graph from file and extract nodes and edges again

In [5]:
road_network = ox.load_graphml(cache_dir / f"network/drive_graph_{place_name}.graphml")

In [6]:
if undirected_graph:
    road_network = road_network.to_undirected()

In [7]:
## 1. convert the graph to GeoDataFrame
road_network_nodes, road_network_edges = ox.graph_to_gdfs(road_network)

In [8]:
print(f"number of nodes in all graph: {len(road_network)}")

number of nodes in all graph: 5718


#### Calculate the intersection of the bomb evacuation area with the graph

In [9]:
# identify the evacuation areas / edges
mask_affected_edges = road_network_edges.intersects(evacuation_area) 
affected_edges = road_network_edges[mask_affected_edges]

In [10]:
# identify the evacuation nodes
mask_affected_nodes = road_network_nodes.intersects(evacuation_area) 
affected_nodes = road_network_nodes[mask_affected_nodes]

#### Validation of the evacuation area

In [11]:
# affected_edges = gdf_edges_drive_graph_trier_default[mask_flooded_edges_trier]
affected_edges_ids = set(affected_edges.index.values)
# un_affected_edges = gdf_edges_drive_graph_trier_default[~mask_flooded_edges_trier]
un_affected_edges = road_network_edges.loc[
    ~road_network_edges.index.isin(affected_edges_ids)
]
if len(road_network_edges) == len(affected_edges) + len(un_affected_edges):
    print(
        f"Edges are split correctly: Affected edges = {len(affected_edges)} unaffected edges = {len(un_affected_edges)} total = {len(road_network_edges)}"
    )
else:
    raise RuntimeError("edges are not correctly split")

Edges are split correctly: Affected edges = 340 unaffected edges = 6715 total = 7055


In [12]:
## create network from gdfs i.e. nodes and edges
# uses all nodes now to avoid any problems with graph / route calculation
road_network_remaining = ox.graph_from_gdfs(road_network_nodes, un_affected_edges)
# road_network_removed can be used for visualization - not relevant for the calculation
road_network_removed = ox.graph_from_gdfs(road_network_nodes, affected_edges)

In [13]:
if undirected_graph:
    road_network_remaining = road_network_remaining.to_undirected()

In [14]:
ox.save_graphml(
    road_network_removed,
    filepath=cache_dir / f"network/drive_graph_removed{place_name}.graphml",
)
ox.save_graphml(
    road_network_remaining,
    filepath=cache_dir / f"network/drive_graph_remaining_{place_name}.graphml",
)